In [32]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score


In [33]:
train_path = "../data/data_preprocess_rf_train_v2.csv"
test_path = "../data/data_preprocess_rf_test_v2.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)


In [34]:
df_test

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,customer_id
0,43,53,25587.55,639,24,106.61,781.55,7.3,0,0,...,2024,9,1,514,0.444444,0.049974,0.0,0.000000,0.136592,CUST_E5RX1BC9II
1,22,5,45378.40,699,21,54.17,252.95,4.2,1,0,...,2025,6,6,243,3.500000,0.014321,0.0,0.000000,-1.356902,CUST_BHWIUKERGN
2,42,2,36643.70,765,17,44.40,238.79,7.0,1,0,...,2024,8,6,551,5.666667,0.014535,0.0,0.000000,-1.356902,CUST_EXT9NA4CHU
3,39,20,30283.30,573,29,38.30,321.44,28.3,0,0,...,2025,5,3,288,1.380952,0.015171,0.0,0.016949,-0.794326,CUST_9FSJE5R1NY
4,54,10,35294.22,624,21,70.93,540.48,21.7,0,0,...,2024,3,0,704,1.909091,0.024108,0.0,0.000000,-2.466131,CUST_GDQXMODBED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,44,3,41531.58,622,24,42.99,214.60,7.3,0,0,...,2025,2,6,362,6.000000,0.012418,0.0,0.020408,-0.794326,CUST_1OM9UCID91
39996,48,12,36783.80,697,25,21.17,69.19,3.8,1,0,...,2024,7,3,596,1.923077,0.006904,0.0,0.000000,-1.356902,CUST_VDEY72BIZP
39997,49,28,55963.60,658,18,180.58,410.17,3.6,2,0,...,2025,12,3,71,0.620690,0.038713,0.0,0.000000,-0.247672,CUST_UQEZ9KKIFG
39998,31,7,9638.61,683,19,129.93,643.13,13.9,0,0,...,2025,5,4,280,2.375000,0.131449,0.0,0.000000,-2.466131,CUST_IXX0BE9VQD


In [ ]:
TARGET_COL = "target_is_fraud"
ID_COL = "customer_id"

LEAKAGE_COLS = [
    "chargeback_resolution_time_days",
    "post_event_status_code"
]

DROP_COLS = [
    "target_is_fraud",
    "customer_id",  
    "chargeback_resolution_time_days",
    "post_event_status_code"
]

df_train = df_train.drop(columns=[c for c in LEAKAGE_COLS if c in df_train.columns])
df_test = df_test.drop(columns=[c for c in LEAKAGE_COLS if c in df_test.columns])


In [70]:
X = df_train.drop(columns=[c for c in DROP_COLS if c in df_train.columns])
y = df_train[TARGET_COL]



In [71]:
df_test

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,customer_id
0,43,53,25587.55,639,24,106.61,781.55,7.3,0,0,...,2024,9,1,514,0.444444,0.049974,0.0,0.000000,0.136592,CUST_E5RX1BC9II
1,22,5,45378.40,699,21,54.17,252.95,4.2,1,0,...,2025,6,6,243,3.500000,0.014321,0.0,0.000000,-1.356902,CUST_BHWIUKERGN
2,42,2,36643.70,765,17,44.40,238.79,7.0,1,0,...,2024,8,6,551,5.666667,0.014535,0.0,0.000000,-1.356902,CUST_EXT9NA4CHU
3,39,20,30283.30,573,29,38.30,321.44,28.3,0,0,...,2025,5,3,288,1.380952,0.015171,0.0,0.016949,-0.794326,CUST_9FSJE5R1NY
4,54,10,35294.22,624,21,70.93,540.48,21.7,0,0,...,2024,3,0,704,1.909091,0.024108,0.0,0.000000,-2.466131,CUST_GDQXMODBED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,44,3,41531.58,622,24,42.99,214.60,7.3,0,0,...,2025,2,6,362,6.000000,0.012418,0.0,0.020408,-0.794326,CUST_1OM9UCID91
39996,48,12,36783.80,697,25,21.17,69.19,3.8,1,0,...,2024,7,3,596,1.923077,0.006904,0.0,0.000000,-1.356902,CUST_VDEY72BIZP
39997,49,28,55963.60,658,18,180.58,410.17,3.6,2,0,...,2025,12,3,71,0.620690,0.038713,0.0,0.000000,-0.247672,CUST_UQEZ9KKIFG
39998,31,7,9638.61,683,19,129.93,643.13,13.9,0,0,...,2025,5,4,280,2.375000,0.131449,0.0,0.000000,-2.466131,CUST_IXX0BE9VQD


In [72]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


In [73]:
num_fraud = y_train.sum()
num_non_fraud = len(y_train) - num_fraud

scale_pos_weight = num_non_fraud / num_fraud


In [ ]:
model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)


{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.8}

In [75]:
X_train

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,partner_risk_indicator,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score
73786,48,0,48390.11,810,25,76.41,145.74,18.6,0,0,...,0.016274,2025,8,4,182,23.000000,0.018944,0.000000,0.019608,-0.778231
133791,40,3,20920.65,698,21,13.95,47.55,1.9,1,0,...,0.016274,2025,6,0,235,5.250000,0.007997,0.000000,0.046512,4.629894
79248,25,6,19811.27,756,20,39.89,131.37,23.7,3,1,...,0.016274,2024,6,1,619,2.857143,0.024147,0.012346,0.024390,6.807003
53549,33,17,32596.38,738,22,117.02,361.23,16.3,1,0,...,0.016274,2024,11,0,452,1.222222,0.043064,0.000000,0.022222,0.338829
19330,55,17,14009.70,708,23,62.85,319.28,6.4,0,0,...,0.016274,2024,9,1,528,1.277778,0.053788,0.000000,0.000000,-2.472897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140157,29,16,52513.98,618,22,51.96,395.97,3.3,0,0,...,0.016274,2024,7,2,590,1.294118,0.011871,0.000000,0.022222,-0.778231
149926,57,5,18849.19,709,23,47.88,326.43,15.9,0,1,...,0.016274,2024,4,0,690,3.833333,0.030463,0.010753,0.021277,3.705256
30146,51,11,35044.92,601,21,60.09,240.17,4.2,2,0,...,0.016274,2024,5,2,660,1.750000,0.020569,0.000000,0.000000,-0.238777
17668,45,2,14503.80,637,27,28.02,235.67,0.7,0,0,...,0.016274,2025,9,2,163,9.000000,0.023164,0.000000,0.000000,-2.472897


In [76]:
model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=0.1, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=600, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

In [77]:
val_proba = model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_proba)
pr_auc = average_precision_score(y_val, val_proba)

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")


ROC-AUC : 0.8464
PR-AUC  : 0.4077


In [78]:
test_ids = df_test[ID_COL]
X_test = df_test.drop(columns=[ID_COL])

test_proba = model.predict(X_test)


In [79]:
submission = pd.DataFrame({
    "customer_id": test_ids,
    "target_is_fraud": test_proba
})

submission.to_csv("../Preds/XGboost_preds_Noah_v3.csv", index=False)


In [80]:
submission

,customer_id,target_is_fraud
0,CUST_E5RX1BC9II,0
1,CUST_BHWIUKERGN,0
2,CUST_EXT9NA4CHU,0
3,CUST_9FSJE5R1NY,0
4,CUST_GDQXMODBED,0
...,...,...
39995,CUST_1OM9UCID91,0
39996,CUST_VDEY72BIZP,0
39997,CUST_UQEZ9KKIFG,0
39998,CUST_IXX0BE9VQD,1


In [86]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "max_depth": [3, 4, 5, 6, 7],
    "min_child_weight": [1, 3, 5, 10, 20],
    "gamma": [0, 0.05, 0.1, 0.3, 1],
    "subsample": [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "reg_alpha": [0, 0.1, 0.5, 1],
    "reg_lambda": [0.5, 1, 3, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1]
}

base_model = xgb.XGBClassifier(
    n_estimators=1000,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=50,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print(search.best_params_)


Fitting 3 folds for each of 50 candidates, totalling 150 fits
{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.8}
